# 03 — Labeling

Create long-only binary triple-barrier labels and validate label quality before training.

In [ ]:
import torch, sys
assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime → T4"
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_BASE  = '/content/drive/MyDrive/yeniBot'
DATA_DIR    = f'{DRIVE_BASE}/data'
CHECKPT_DIR = f'{DRIVE_BASE}/checkpoints'
os.makedirs(f'{DATA_DIR}/processed', exist_ok=True)
os.makedirs(CHECKPT_DIR, exist_ok=True)

In [ ]:
import os, sys
REPO_URL = 'https://github.com/umutergul74/yeniBot.git'
REPO_DIR = '/content/yenibot_repo'
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
sys.path.insert(0, REPO_DIR)
print('After git pull, use Runtime → Restart session before trusting changed imports.')

In [ ]:
!pip install -q -r {REPO_DIR}/requirements.txt

In [ ]:
import yaml
with open(f'{REPO_DIR}/config.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['paths']['data_dir'] = DATA_DIR
cfg['paths']['checkpoint_dir'] = CHECKPT_DIR

In [ ]:
import pandas as pd
from yenibot.labeling import add_long_only_labels, validate_label_quality

features = pd.read_parquet(f'{DATA_DIR}/processed/features_1h.parquet')
label_cfg = cfg['labeling']
labeled = add_long_only_labels(
    features,
    atr_column=label_cfg['atr_column'],
    tp_multiplier=label_cfg['tp_multiplier'],
    sl_multiplier=label_cfg['sl_multiplier'],
    max_holding_bars=label_cfg['max_holding_bars'],
)
quality = validate_label_quality(
    labeled,
    min_long_forward_return=label_cfg['min_long_forward_return'],
    max_not_long_forward_return=label_cfg['max_not_long_forward_return'],
    min_long_pct=label_cfg['min_long_pct'],
    max_long_pct=label_cfg['max_long_pct'],
)
print(quality)
print(labeled['label'].value_counts(normalize=True).sort_index())
print(labeled['hit_type'].value_counts(normalize=True))
out = f'{DATA_DIR}/processed/labeled_1h.parquet'
labeled.to_parquet(out, index=False)
print('Saved', out)

In [ ]:
import matplotlib.pyplot as plt
labeled.groupby('label')['fwd_return_10h'].mean().plot(kind='bar', title='Forward return by class')
plt.show()
labeled.set_index('timestamp')['label'].rolling(168).mean().plot(title='Rolling long-label rate')
plt.show()

In [ ]:
from google.colab import runtime
runtime.unassign()